# Dispersion Intuition

This notebook builds intuition for the echelle grating equation and pixel
dispersion using the low-level functions in `echelle_optics.grating`.

## The grating equation

$$m \lambda = d (\sin\alpha + \sin\beta)$$

$d$ = groove spacing, $\alpha$ = incidence angle, $\beta$ = diffraction angle,
$m$ = diffraction order.

## Quasi-Littrow approximation

For a quasi-Littrow echelle $\alpha \approx \beta \approx \theta_B$ (blaze angle), so

$$m \lambda \approx 2 d \sin\theta_B \equiv K \quad\Rightarrow\quad \lambda_{\text{center}} = K / m$$

$K$ is the *Littrow constant* — a single number that sets the entire order-wavelength ladder.

## Linear dispersion

Differentiating the grating equation with respect to detector position $x$:

$$\frac{d\lambda}{dx} = \frac{d\cos\beta}{m f}$$

Per pixel ($p$ = pixel size, $f$ = focal length):

$$\frac{d\lambda}{\text{dpx}} = \frac{d \cos\theta_B \cdot p}{m f}$$

This is strictly proportional to $1/m$, so a plot of dispersion vs $1/m$
should be a straight line through the origin.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import linregress

from echelle_optics.grating import (
    groove_spacing_nm,
    littrow_constant_nm,
    central_wavelength_nm,
    physical_order_from_wavelength,
    linear_dispersion_nm_per_px,
    free_spectral_range_nm,
)

# LHD CMOS echelle parameters
GROOVES = 46.1
BLAZE   = 32.0
F       = 304.8  # mm
PX      = 6.5    # µm

In [2]:
d_nm = groove_spacing_nm(GROOVES)
K    = littrow_constant_nm(GROOVES, BLAZE)

print(f"Groove spacing d = {d_nm:.2f} nm  ({d_nm/1000:.4f} µm)")
print(f"Littrow constant K = {K:.1f} nm")
print()

print(f"{'Order':>6}  {'λ_center (nm)':>15}  {'FSR (nm)':>10}  {'dλ/dpx (pm/px)':>16}")
print("-" * 55)
for m in range(30, 59, 4):
    lam = central_wavelength_nm(m, GROOVES, BLAZE)
    fsr = free_spectral_range_nm(lam, m)
    disp = linear_dispersion_nm_per_px(m, GROOVES, BLAZE, F, PX)
    print(f"{m:>6}  {lam:>15.2f}  {fsr:>10.3f}  {disp*1000:>16.4f}")

Groove spacing d = 21691.97 nm  (21.6920 µm)
Littrow constant K = 22990.0 nm

 Order    λ_center (nm)    FSR (nm)    dλ/dpx (pm/px)
-------------------------------------------------------
    30           766.33      25.544           13.0767
    34           676.18      19.888           11.5382
    38           605.00      15.921           10.3237
    42           547.38      13.033            9.3405
    46           499.78      10.865            8.5283
    50           459.80       9.196            7.8460
    54           425.74       7.884            7.2648
    58           396.38       6.834            6.7638


In [3]:
# Physical order from wavelength
test_wavelengths = [400, 450, 500, 550, 600, 650, 700]
print("Wavelength → physical order (fractional):")
for lam in test_wavelengths:
    m_frac = physical_order_from_wavelength(lam, GROOVES, BLAZE)
    print(f"  {lam} nm  →  m = {m_frac:.2f}  (nearest integer: {round(m_frac)})")

Wavelength → physical order (fractional):
  400 nm  →  m = 57.47  (nearest integer: 57)
  450 nm  →  m = 51.09  (nearest integer: 51)
  500 nm  →  m = 45.98  (nearest integer: 46)
  550 nm  →  m = 41.80  (nearest integer: 42)
  600 nm  →  m = 38.32  (nearest integer: 38)
  650 nm  →  m = 35.37  (nearest integer: 35)
  700 nm  →  m = 32.84  (nearest integer: 33)


## Dispersion vs 1/m — linearity check

Theory predicts $d\lambda/\text{dpx} = S / m$ where
$S = d \cos\theta_B \cdot p / f$.  The slope of the
dispersion-vs-$1/m$ line should equal $S$.

In [4]:
orders = np.arange(30, 59)
inv_m  = 1.0 / orders
disp   = np.array([
    linear_dispersion_nm_per_px(m, GROOVES, BLAZE, F, PX) for m in orders
])

# Force fit through origin: slope = sum(x*y)/sum(x*x)
slope = np.dot(inv_m, disp) / np.dot(inv_m, inv_m)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(inv_m, disp * 1000, "o", ms=5, label="computed dispersion")
ax.plot(inv_m, slope * 1000 * inv_m, "--", lw=1.5,
        label=f"fit: slope = {slope*1000:.4f} pm/px")
ax.set_xlabel("1 / order")
ax.set_ylabel("dλ/dpx (pm/px)")
ax.set_title("Dispersion scales linearly with 1/m")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Fitted slope S = {slope:.6f} nm/px  (expected ≈ 0.392 nm/px)")

<Figure size 700x400 with 1 Axes>

Fitted slope S = 0.392300 nm/px  (expected ≈ 0.392 nm/px)


The slope $S \approx 0.392$ nm/px matches the predicted value:

$$S = \frac{d \cos\theta_B \cdot p}{f}
    = \frac{21\,690\,\text{nm} \times 0.848 \times 0.0065\,\text{mm}}{304.8\,\text{mm}}
    \approx 0.392\,\text{nm/px}$$

This confirms the dispersion model is self-consistent.